# بک‌تست کانال BITFA — فاز ۱: ارزیابی داده‌ی خام

این نوت‌بوک ۴۵ ارزی که در کانال معرفی شدن رو تحلیل می‌کنه.

**هشدار مهم:** عدد `reported_gain_pct` همون چیزیه که در متن کانال به‌صورت «تا تاریخ X، Y% سود داده» اومده — این معمولاً **بیشترین قیمت رسیده در بازه** است، نه سود واقعیِ یک معامله‌ی واقعی با قانون خروج مشخص. در سلول‌های بعدی این مشکل رو با گرفتن قیمت واقعی روزانه از Binance/MEXC اصلاح می‌کنیم.

In [ ]:
!pip install -q pandas requests
import pandas as pd
import requests
from datetime import datetime


In [ ]:
# داده‌ی استخراج‌شده از پست‌های کانال (symbol, announce_date, reported_gain_pct, status) -- الان ۷۲ نمونه
raw = [('VVV', '2026-07-21', 33, 'win'), ('BR', '2026-08-12', 33, 'win'), ('RE', '2026-08-19', 54, 'win'), ('LIT', '2026-07-27', 52, 'win'), ('MORPHO', '2026-08-18', 22, 'win'), ('ZRO', '2026-07-17', 61, 'win'), ('PUMP', '2026-07-19', 175, 'win'), ('HEMI', '2026-08-15', 61, 'win'), ('BTW', '2026-07-27', 670, 'win'), ('Q', '2026-08-16', None, 'loss'), ('CYS', '2026-08-05', 212, 'win'), ('BLUAI', '2026-07-06', 204, 'win'), ('TST', '2026-08-10', 20, 'win'), ('MUBARAK', '2026-08-09', 58, 'win'), ('BROCCOLI', '2026-07-31', 42, 'win'), ('EPIC', '2026-07-11', 52, 'win'), ('US', '2026-07-10', 130, 'win'), ('ON', '2026-07-28', 50, 'win'), ('ZBT', '2026-08-06', 21, 'win'), ('KOMA', '2026-07-31', 40, 'win'), ('GIGGLE', '2026-07-31', 50, 'win'), ('ONDO', '2026-07-31', None, 'loss'), ('KGEN', '2026-07-31', None, 'loss'), ('UAI', '2026-07-30', None, 'loss'), ('ESP', '2026-07-19', 60, 'win'), ('TAG', '2026-07-27', None, 'loss'), ('KAITO', '2026-07-15', 68, 'win'), ('ZAMA', '2026-07-20', 48, 'win'), ('STAR', '2026-07-19', None, 'loss'), ('ALLO', '2026-06-08', 12, 'win'), ('SAGA', '2026-09-15', 90, 'win'), ('ENA', '2026-08-22', 51, 'win'), ('AKE', '2026-09-02', 955, 'win'), ('NEAR', '2026-09-17', 58, 'win'), ('XTZ', '2026-09-19', 18, 'win'), ('PIEVERSE', '2026-08-08', 116, 'win'), ('RAY', '2026-09-06', 66, 'win'), ('MOO', '2026-09-11', 97, 'win'), ('COTI', '2026-09-07', 55, 'win'), ('PONS', '2026-08-24', 1470, 'win'), ('LOBSTER', '2026-08-29', 418, 'win'), ('AVA', '2026-09-13', 65, 'win'), ('KSM', '2026-09-17', None, 'pending'), ('ZEC', '2026-08-21', 160, 'win'), ('ONE', '2026-09-17', 208, 'win'), ('UNI', '2026-09-05', 54, 'win'), ('MARSCOIN', '2026-09-22', 20, 'win'), ('FLOCK', '2026-08-31', 150, 'win'), ('MINA', '2026-09-21', 20, 'win'), ('PYTH', '2026-09-02', 20, 'win'), ('KERNEL', '2026-08-31', 90, 'win'), ('PEPE', '2026-09-21', None, 'pending'), ('KMNO', '2026-09-20', 18, 'win'), ('ONDO', '2026-09-19', None, 'loss'), ('NEET', '2026-08-27', None, 'loss'), ('MAGMA', '2026-08-28', 43, 'win'), ('MANCER', '2026-08-30', 110, 'win'), ('RDDT', '2026-08-31', None, 'loss'), ('ANTFUN', '2026-08-31', None, 'loss'), ('USELESS', '2026-08-31', 260, 'win'), ('BONK', '2026-09-01', None, 'loss'), ('LINK', '2026-09-01', None, 'loss'), ('BTR', '2026-09-01', None, 'loss'), ('FF', '2026-08-27', 90, 'win'), ('UAI', '2026-08-30', 123, 'win'), ('MICRODUCK', '2026-09-02', None, 'loss'), ('TENDIES', '2026-09-02', None, 'loss'), ('KITE', '2026-09-02', None, 'loss'), ('INDEX', '2026-09-02', 80, 'win'), ('WTH', '2026-09-03', None, 'loss'), ('USELESS', '2026-09-03', 260, 'win'), ('MARSCOIN', '2026-09-03', 90, 'win')]
df = pd.DataFrame(raw, columns=["symbol","announce_date","reported_gain_pct","status"])
df["announce_date"] = pd.to_datetime(df["announce_date"])
df

## آمار پایه (بر اساس داده‌ی خامِ کانال — هنوز اصلاح‌نشده)

In [ ]:
total = len(df)
wins = (df["status"]=="win").sum()
losses = (df["status"]=="loss").sum()
pending = (df["status"]=="pending").sum()

print(f"تعداد کل: {total}")
print(f"برد (win): {wins}  ({wins/total*100:.1f}%)")
print(f"باخت (loss / no-mention): {losses}  ({losses/total*100:.1f}%)")
print(f"در انتظار (pending): {pending}")

win_df = df[df["status"]=="win"]
print()
print("میانگین سود بردها:", round(win_df["reported_gain_pct"].mean(),1), "%")
print("میانه سود بردها:", round(win_df["reported_gain_pct"].median(),1), "%")
print("بیشترین سود:", win_df["reported_gain_pct"].max(), "%  -->", win_df.loc[win_df["reported_gain_pct"].idxmax(),"symbol"])


## نکته: میانگین vs میانه

چون چند ارز (AKE=955%، PONS=1470%، BTW=670%) بازدهی خیلی دورافتاده دارن، میانگین گمراه‌کننده‌ست. میانه واقعی‌تره. اگه اون سه ارز حذف بشن، میانگین چقدر تغییر می‌کنه؟

In [ ]:
outliers = ["AKE","PONS","BTW"]
win_no_outliers = win_df[~win_df["symbol"].isin(outliers)]
print("میانگین بدون ۳ ارز پرت:", round(win_no_outliers["reported_gain_pct"].mean(),1), "%")


## فاز بعدی — گرفتن قیمت واقعی از Binance (اسپات)

چون روی صرافی‌های اسپات (Binance/MEXC) معامله می‌کنید، از API عمومی Binance برای قیمت‌های روزانه استفاده می‌کنیم — رایگان و بدون نیاز به کلید.

In [ ]:
def get_binance_daily(symbol_pair, start_date, days=10):
    """
    قیمت close روزانه رو از Binance برای symbol_pair (مثل 'ONDOUSDT') می‌گیره.
    start_date: string مثل '2026-07-31'
    """
    start_ts = int(pd.Timestamp(start_date).timestamp() * 1000)
    url = "https://api.binance.com/api/v3/klines"
    params = {
        "symbol": symbol_pair,
        "interval": "1d",
        "startTime": start_ts,
        "limit": days
    }
    try:
        r = requests.get(url, params=params, timeout=10)
    except Exception as e:
        print("خطای اتصال:", e)
        return None
    if r.status_code != 200:
        print("خطای Binance -- status:", r.status_code, "| پاسخ:", r.text[:200])
        return None
    rows = r.json()
    closes = [float(row[4]) for row in rows]
    return closes


def get_mexc_daily(symbol_pair, start_date, days=10):
    """
    همون کار get_binance_daily رو برای MEXC انجام می‌ده -- فرمت API یکسانه.
    symbol_pair مثل 'ONDOUSDT'
    """
    start_ts = int(pd.Timestamp(start_date).timestamp() * 1000)
    url = "https://api.mexc.com/api/v3/klines"
    params = {
        "symbol": symbol_pair,
        "interval": "1d",
        "startTime": start_ts,
        "limit": days
    }
    try:
        r = requests.get(url, params=params, timeout=10)
    except Exception as e:
        print("خطای اتصال:", e)
        return None
    if r.status_code != 200:
        print("خطای MEXC -- status:", r.status_code, "| پاسخ:", r.text[:200])
        return None
    rows = r.json()
    closes = [float(row[4]) for row in rows]
    return closes


# تست روی یک نماد شناخته‌شده -- هر دو صرافی رو امتحان می‌کنیم
print("--- Binance ---")
test_b = get_binance_daily("ONDOUSDT", "2026-07-31", days=10)
print(test_b)

print("--- MEXC ---")
test_m = get_mexc_daily("ONDOUSDT", "2026-07-31", days=10)
print(test_m)


## اسکلت شبیه‌سازی خروج ثابت (فاز بعد — بعد از تست دسترسی API)

بعد از این‌که مطمئن شدیم کدوم نمادها روی Binance/MEXC موجودن، این تابع رو تکمیل می‌کنیم:

In [ ]:
def simulate_fixed_exit(closes, entry_price, target_pct=50, stop_pct=-15):
    """
    closes: لیست قیمت‌های close روزانه بعد از ورود
    برمی‌گردونه: (نتیجه, درصد سود/ضرر واقعی, چندمین روز)
    """
    for i, price in enumerate(closes):
        change = (price - entry_price) / entry_price * 100
        if change >= target_pct:
            return ("target_hit", round(change,1), i+1)
        if change <= stop_pct:
            return ("stop_hit", round(change,1), i+1)
    # اگه نه تارگت نه استاپ خورد، آخرین قیمت رو برمی‌گردونیم
    if closes:
        final_change = (closes[-1] - entry_price) / entry_price * 100
        return ("held_to_end", round(final_change,1), len(closes))
    return ("no_data", None, 0)


## اجرای کامل روی هر ۴۵ ارز (با MEXC)

چون Binance مسدوده، از MEXC به‌عنوان منبع اصلی استفاده می‌کنیم. نکته: قیمت ورود رو اولین close روزانه‌ی بعد از تاریخ اعلام در نظر می‌گیریم (چون قیمت لحظه‌ای اعلام رو نداریم -- این خودش یه تقریبه، نه دقیق).

بعضی نمادها (مخصوصاً میم‌کوین‌های خیلی جدید یا نمادهای عمومی مثل US/ON/Q) ممکنه اصلاً روی MEXC با جفت USDT لیست نباشن -- این‌ها به‌صورت `no_pair` علامت می‌خورن و باید دستی چک بشن.

In [ ]:
import time

results = []
for _, row in df.iterrows():
    symbol = row["symbol"]
    pair = symbol + "USDT"
    announce_date = row["announce_date"].strftime("%Y-%m-%d")

    closes = get_mexc_daily(pair, announce_date, days=10)
    time.sleep(0.3)  # جلوگیری از rate limit

    if not closes:
        results.append({
            "symbol": symbol,
            "reported_gain_pct": row["reported_gain_pct"],
            "reported_status": row["status"],
            "real_outcome": "no_pair_or_no_data",
            "real_pct": None,
            "days_to_exit": None,
        })
        continue

    entry_price = closes[0]
    outcome, real_pct, day_n = simulate_fixed_exit(closes[1:], entry_price, target_pct=50, stop_pct=-15)

    results.append({
        "symbol": symbol,
        "reported_gain_pct": row["reported_gain_pct"],
        "reported_status": row["status"],
        "real_outcome": outcome,
        "real_pct": real_pct,
        "days_to_exit": day_n,
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
# مقایسه‌ی نرخ برد گزارش‌شده در کانال با نرخ برد واقعی (با قانون خروج ثابت +۵۰٪ / -۱۵٪)
valid = results_df[results_df["real_outcome"] != "no_pair_or_no_data"]
target_hits = (valid["real_outcome"] == "target_hit").sum()
stop_hits = (valid["real_outcome"] == "stop_hit").sum()
held = (valid["real_outcome"] == "held_to_end").sum()
no_pair = (results_df["real_outcome"] == "no_pair_or_no_data").sum()

print(f"تعداد ارزهایی که روی MEXC پیدا شدن: {len(valid)} از ۴۵")
print(f"پیدا نشد (no_pair): {no_pair}")
print()
print(f"رسیدن به تارگت +۵۰٪: {target_hits}  ({target_hits/len(valid)*100:.1f}%)")
print(f"خوردن استاپ -۱۵٪: {stop_hits}  ({stop_hits/len(valid)*100:.1f}%)")
print(f"نه تارگت نه استاپ (تا آخر بازه نگه‌داشته شد): {held}  ({held/len(valid)*100:.1f}%)")
print()
print("میانگین سود واقعی (فقط target_hit ها):", round(valid[valid["real_outcome"]=="target_hit"]["real_pct"].mean(),1), "%")

## ماژول ۳ — امتیازدهی (Confidence Score) — معماری پلاگینی

هر اندیکاتور یه تابع مستقله که یه عدد بین ۰ تا ۱ برمی‌گردونه (چقدر این سیگنال مثبته). بعداً هر اندیکاتور جدید (FVG، ICT، Ichimoku، Smart Money...) فقط باید یه تابع جدید با همین امضا اضافه بشه و در دیکشنری `PLUGINS` ثبت بشه -- بدون دست‌زدن به بقیه‌ی کد.

In [ ]:
def get_mexc_full_klines(symbol_pair, days=30):
    """
    برخلاف get_mexc_daily، اینجا OHLCV کامل رو برای N روز اخیر (تا امروز) می‌گیریم --
    برای محاسبه‌ی اندیکاتورهایی مثل RSI به تاریخچه‌ی بیشتر از لحظه‌ی سیگنال نیاز داریم.
    نکته: فرمت کندل MEXC با Binance فرق داره -- MEXC فقط 8 ستون برمی‌گردونه، نه 12.
    """
    url = "https://api.mexc.com/api/v3/klines"
    params = {"symbol": symbol_pair, "interval": "1d", "limit": days}
    try:
        r = requests.get(url, params=params, timeout=10)
    except Exception as e:
        print("خطای اتصال:", e)
        return None
    if r.status_code != 200:
        print(f"خطای MEXC ({symbol_pair}) -- status:", r.status_code, r.text[:150])
        return None
    rows = r.json()
    if not rows:
        return None
    n_cols = len(rows[0])
    base_cols = ["open_time","open","high","low","close","volume","close_time","quote_vol"]
    extra_cols = ["trades","taker_base","taker_quote","ignore"]
    columns = (base_cols + extra_cols)[:n_cols]
    df_k = pd.DataFrame(rows, columns=columns)
    for c in ["open","high","low","close","volume"]:
        df_k[c] = df_k[c].astype(float)
    return df_k


In [ ]:
# --- پلاگین ۱: شکست/تثبیت مقاومت (خط روند) ---
def plugin_trend_break(ohlcv: pd.DataFrame) -> float:
    """
    اگه قیمت فعلی نزدیک یا بالای سقف N روز اخیر باشه (شکست مقاومت)، امتیاز بالا می‌گیره.
    """
    if ohlcv is None or len(ohlcv) < 5:
        return 0.5  # داده کافی نیست -- خنثی
    recent = ohlcv.iloc[:-1]  # همه به‌جز آخرین کندل
    resistance = recent["high"].max()
    current_price = ohlcv["close"].iloc[-1]
    if current_price >= resistance:
        return 1.0
    ratio = current_price / resistance
    return max(0.0, min(1.0, (ratio - 0.9) / 0.1))  # هرچی به مقاومت نزدیک‌تر، امتیاز بالاتر (از ۹۰٪ به بالا)


# --- پلاگین ۲: حجم معاملات نسبت به میانگین ---
def plugin_volume_spike(ohlcv: pd.DataFrame) -> float:
    """
    اگه حجم روز/کندل آخر بیشتر از میانگین ۱۰ روزه باشه، یعنی حرکت واقعی پشتش هست.
    """
    if ohlcv is None or len(ohlcv) < 10:
        return 0.5
    avg_vol = ohlcv["volume"].iloc[-11:-1].mean()
    last_vol = ohlcv["volume"].iloc[-1]
    if avg_vol == 0:
        return 0.5
    ratio = last_vol / avg_vol
    return max(0.0, min(1.0, ratio / 3))  # ۳ برابر میانگین یا بیشتر = امتیاز کامل


# --- پلاگین ۳: RSI ---
def compute_rsi(closes: pd.Series, period=14) -> float:
    delta = closes.diff()
    gain = delta.clip(lower=0).rolling(period).mean()
    loss = (-delta.clip(upper=0)).rolling(period).mean()
    rs = gain / loss.replace(0, 1e-9)
    rsi = 100 - (100 / (1 + rs))
    return rsi.iloc[-1]

def plugin_rsi(ohlcv: pd.DataFrame) -> float:
    """
    RSI بین ۴۰ تا ۷۰ بهترین حالته (نه اشباع خرید، نه اشباع فروش).
    زیر ۳۰ یا بالای ۸۰ امتیاز پایین می‌گیره.
    """
    if ohlcv is None or len(ohlcv) < 15:
        return 0.5
    rsi = compute_rsi(ohlcv["close"])
    if pd.isna(rsi):
        return 0.5
    if 40 <= rsi <= 70:
        return 1.0
    if rsi < 30 or rsi > 80:
        return 0.2
    return 0.6


# --- ثبت پلاگین‌ها -- بعداً FVG/ICT/Ichimoku/Smart Money اینجا اضافه می‌شن ---
PLUGINS = {
    "trend_break": (plugin_trend_break, 0.4),   # (تابع, وزن)
    "volume_spike": (plugin_volume_spike, 0.3),
    "rsi": (plugin_rsi, 0.3),
    # "fvg": (plugin_fvg, 0.0),          # TODO
    # "ichimoku": (plugin_ichimoku, 0.0), # TODO
    # "smart_money": (plugin_smart_money, 0.0), # TODO
}


def score_coin(symbol_pair: str) -> dict:
    ohlcv = get_mexc_full_klines(symbol_pair, days=30)
    breakdown = {}
    total = 0.0
    total_weight = 0.0
    for name, (func, weight) in PLUGINS.items():
        s = func(ohlcv)
        breakdown[name] = round(s, 2)
        total += s * weight
        total_weight += weight
    confidence = round((total / total_weight) * 100, 1) if total_weight else 0.0
    return {"symbol": symbol_pair, "confidence_pct": confidence, "breakdown": breakdown}


In [ ]:
# تست روی چندتا نماد از واچ‌لیست خودت
for sym in ["ONDOUSDT", "MUBARAKUSDT", "SAGAUSDT"]:
    print(score_coin(sym))


### نکته درباره‌ی اندازه‌ی پوزیشن (طبق جوابت: بر اساس امتیاز)

In [ ]:
def position_size_pct(confidence_pct: float) -> float:
    """
    درصد سرمایه‌ای که باید تخصیص داده بشه، بر اساس امتیاز اطمینان.
    قابل تنظیم -- این فقط نسخه‌ی اول است.
    """
    if confidence_pct < 50:
        return 0.0      # رد کامل
    if confidence_pct < 70:
        return 0.15
    if confidence_pct < 85:
        return 0.25
    return 0.35

# مثال با سرمایه‌ی ۶۴ دلاری
capital = 64
for sym in ["ONDOUSDT", "MUBARAKUSDT", "SAGAUSDT"]:
    result = score_coin(sym)
    pct = position_size_pct(result["confidence_pct"])
    print(f'{sym}: امتیاز={result["confidence_pct"]}% -> پوزیشن پیشنهادی={pct*100:.0f}% (${capital*pct:.1f})  | جزئیات: {result["breakdown"]}')


## کالیبراسیون -- اسکور هر ۴۵ ارز رو دقیقاً لحظه‌ی اعلامش حساب می‌کنیم و با نتیجه‌ی واقعی مقایسه می‌کنیم

In [ ]:
def get_mexc_full_klines_at(symbol_pair, end_date, days=30):
    """مثل get_mexc_full_klines ولی داده رو تا یک تاریخ مشخص (لحظه‌ی اعلام) می‌گیره، نه تا امروز."""
    url = "https://api.mexc.com/api/v3/klines"
    end_ts = int(pd.Timestamp(end_date).timestamp() * 1000) + 24*3600*1000
    params = {"symbol": symbol_pair, "interval": "1d", "limit": days, "endTime": end_ts}
    try:
        r = requests.get(url, params=params, timeout=10)
    except Exception:
        return None
    if r.status_code != 200:
        return None
    rows = r.json()
    if not rows:
        return None
    n_cols = len(rows[0])
    columns = (["open_time","open","high","low","close","volume","close_time","quote_vol",
                "trades","taker_base","taker_quote","ignore"])[:n_cols]
    dfk = pd.DataFrame(rows, columns=columns)
    for c in ["open","high","low","close","volume"]:
        dfk[c] = dfk[c].astype(float)
    return dfk


def score_ohlcv(ohlcv):
    breakdown, total, total_weight = {}, 0.0, 0.0
    for name, (func, weight) in PLUGINS.items():
        s = func(ohlcv)
        breakdown[name] = round(s, 2)
        total += s * weight
        total_weight += weight
    confidence = round((total / total_weight) * 100, 1) if total_weight else 0.0
    return confidence, breakdown


In [ ]:
import time

calibration = []
for _, row in df.iterrows():
    symbol = row["symbol"]
    ohlcv = get_mexc_full_klines_at(symbol + "USDT", row["announce_date"].strftime("%Y-%m-%d"))
    if ohlcv is None or len(ohlcv) < 10:
        continue
    confidence, breakdown = score_ohlcv(ohlcv)
    match = results_df[results_df["symbol"] == symbol]
    real_pct = match["real_pct"].values[0] if len(match) else None
    calibration.append({"symbol": symbol, "confidence_pct": confidence, **breakdown, "real_pct": real_pct})
    time.sleep(0.3)

calib_df = pd.DataFrame(calibration)
calib_df

In [ ]:
valid = calib_df.dropna(subset=["real_pct"])
print(valid[["confidence_pct","trend_break","volume_spike","rsi","real_pct"]].corr()["real_pct"])

## اصلاح RSI -- از mean-reversion به momentum (چون همبستگی اولیه منفی بود)

In [ ]:
def plugin_rsi_momentum(ohlcv):
    """نسخه‌ی اصلاح‌شده: به‌جای mean-reversion، منطق momentum -- RSI بالا امتیاز بالا می‌گیره."""
    if ohlcv is None or len(ohlcv) < 15:
        return 0.5
    rsi = compute_rsi(ohlcv["close"])
    if pd.isna(rsi):
        return 0.5
    return max(0.0, min(1.0, (rsi - 30) / 50))  # RSI=30 -> 0 , RSI=80 -> 1

PLUGINS["rsi"] = (plugin_rsi_momentum, 0.2)
PLUGINS["trend_break"] = (plugin_trend_break, 0.5)
PLUGINS["volume_spike"] = (plugin_volume_spike, 0.3)

calibration2 = []
for _, row in df.iterrows():
    symbol = row["symbol"]
    ohlcv = get_mexc_full_klines_at(symbol + "USDT", row["announce_date"].strftime("%Y-%m-%d"))
    if ohlcv is None or len(ohlcv) < 10:
        continue
    confidence, breakdown = score_ohlcv(ohlcv)
    match = results_df[results_df["symbol"] == symbol]
    real_pct = match["real_pct"].values[0] if len(match) else None
    calibration2.append({"symbol": symbol, "confidence_pct": confidence, **breakdown, "real_pct": real_pct})
    time.sleep(0.3)

calib_df2 = pd.DataFrame(calibration2)
valid2 = calib_df2.dropna(subset=["real_pct"])
print(valid2[["confidence_pct","trend_break","volume_spike","rsi","real_pct"]].corr()["real_pct"])

## وزن‌دهی نهایی بر اساس همبستگی واقعی (RSI قوی‌ترین سیگنال بود)

In [ ]:
PLUGINS["rsi"] = (plugin_rsi_momentum, 0.5)     # قوی‌ترین سیگنال
PLUGINS["trend_break"] = (plugin_trend_break, 0.3)
PLUGINS["volume_spike"] = (plugin_volume_spike, 0.2)

calibration3 = []
for _, row in df.iterrows():
    symbol = row["symbol"]
    ohlcv = get_mexc_full_klines_at(symbol + "USDT", row["announce_date"].strftime("%Y-%m-%d"))
    if ohlcv is None or len(ohlcv) < 10:
        continue
    confidence, breakdown = score_ohlcv(ohlcv)
    match = results_df[results_df["symbol"] == symbol]
    real_pct = match["real_pct"].values[0] if len(match) else None
    calibration3.append({"symbol": symbol, "confidence_pct": confidence, "real_pct": real_pct})
    time.sleep(0.3)

calib_df3 = pd.DataFrame(calibration3)
valid3 = calib_df3.dropna(subset=["real_pct"])
print("همبستگی نهایی confidence با real_pct:", valid3["confidence_pct"].corr(valid3["real_pct"]))

valid3["score_bucket"] = pd.cut(valid3["confidence_pct"], bins=[0,40,60,100], labels=["پایین(زیر40)","متوسط(40-60)","بالا(بالای60)"])
print(valid3.groupby("score_bucket")["real_pct"].agg(["mean","count"]))